# Transformers Pipeline

The `pipeline()` function is the highest-level API in HuggingFace Transformers. It handles **tokenization → model inference → post-processing** in a single call, letting you run state-of-the-art models without knowing the internals.

In this notebook we explore 6 different NLP tasks, understand batching for efficiency, and peek inside how a pipeline works.

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. Sentiment Analysis

Classify text as positive or negative. The default model is `distilbert-base-uncased-finetuned-sst-2-english`.

In [ ]:
sentiment = pipeline("sentiment-analysis", device=device)

texts = [
    "I absolutely love working with Hugging Face models!",
    "This API is confusing and the documentation is terrible.",
    "The weather today is neither good nor bad.",
]

for text, result in zip(texts, sentiment(texts)):
    label = result["label"]
    score = result["score"]
    print(f"[{label:8s} {score:.2f}]  {text}")

## 2. Named Entity Recognition (NER)

NER identifies and classifies named entities (persons, organizations, locations, dates) in text.

In [ ]:
ner = pipeline("ner", aggregation_strategy="simple", device=device)

text = "Elon Musk founded SpaceX in 2002 in Hawthorne, California."
entities = ner(text)

print(f"Text: {text}\n")
for ent in entities:
    print(f"  {ent['entity_group']:5s}  '{ent['word']}'  (score: {ent['score']:.2f})")

## 3. Question Answering (Extractive)

Given a **context** paragraph and a **question**, the model extracts the answer span directly from the context.

In [ ]:
qa = pipeline("question-answering", device=device)

context = """
Hugging Face is a machine learning company based in New York City and Paris.
It was founded in 2016 and is best known for its Transformers library,
which provides thousands of pre-trained models for NLP, vision, and audio tasks.
"""

questions = [
    "Where is Hugging Face based?",
    "When was Hugging Face founded?",
    "What is Hugging Face best known for?",
]

for q in questions:
    result = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (score: {result['score']:.2f})\n")

## 4. Text Summarization

Condense a long document to a short summary. The default model (`sshleifer/distilbart-cnn-12-6`) is fast and accurate for news-style text.

In [ ]:
summarizer = pipeline("summarization", device=device)

article = """
The development of large language models (LLMs) has transformed the field of natural
language processing. Models such as GPT-4, Claude, and Gemini can generate coherent
text, answer complex questions, write code, and perform reasoning tasks that were
previously thought to require human intelligence. These models are trained on massive
datasets scraped from the internet, comprising hundreds of billions of tokens of text.
Training requires thousands of GPUs running for weeks or months. Despite their
impressive capabilities, LLMs still face challenges such as hallucination (generating
false information confidently), difficulty with long-term reasoning, and high
computational costs for inference.
"""

summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
print("Original length:", len(article.split()), "words")
print("Summary length :", len(summary[0]["summary_text"].split()), "words")
print("\nSummary:", summary[0]["summary_text"])

## 5. Translation

Translate between languages using sequence-to-sequence models (Helsinki-NLP Opus-MT family).

In [ ]:
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr", device=device)

sentences = [
    "Machine learning is revolutionizing how we build software.",
    "The Hugging Face Hub hosts thousands of pre-trained models.",
    "Deep learning models learn representations from raw data.",
]

translations = translator(sentences)
print("English → French\n")
for src, tgt in zip(sentences, translations):
    print(f"  EN: {src}")
    print(f"  FR: {tgt['translation_text']}\n")

## 6. Zero-Shot Classification

Classify text into **any categories you define** — no task-specific training needed. The model reasons about whether the text entails each label.

In [ ]:
zero_shot = pipeline("zero-shot-classification", device=device)

texts_and_labels = [
    ("The new iPhone 15 features a titanium chassis and USB-C connector.", ["technology", "sports", "politics", "finance"]),
    ("The central bank raised interest rates by 25 basis points today.", ["technology", "sports", "politics", "finance"]),
    ("The team won the championship after a dramatic penalty shootout.", ["technology", "sports", "politics", "finance"]),
]

for text, labels in texts_and_labels:
    result = zero_shot(text, candidate_labels=labels)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    print(f"Text  : {text[:60]}...")
    print(f"Label : {top_label} ({top_score:.2f})\n")

## How Pipelines Work Internally

Every `pipeline()` call does three things under the hood:

```
Raw text
   │
   ▼
[Tokenizer]  → converts text to token IDs (integers)
   │
   ▼
[Model]      → runs transformer forward pass, returns logits/embeddings
   │
   ▼
[Post-processor] → converts model output to human-readable result
```

You can access each component directly:

In [ ]:
# Peek inside a pipeline
pipe = pipeline("sentiment-analysis", device=device)

tokenizer = pipe.tokenizer
model     = pipe.model

text = "Transformers are incredibly powerful!"

# Step 1: tokenize
tokens = tokenizer(text, return_tensors="pt")
print("Token IDs :", tokens["input_ids"])
print("Tokens    :", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

# Step 2: model forward pass
import torch
with torch.no_grad():
    outputs = model(**tokens)

print("\nRaw logits:", outputs.logits)
probs = torch.softmax(outputs.logits, dim=-1)
labels = model.config.id2label
print("Probabilities:", {labels[i]: f"{p:.3f}" for i, p in enumerate(probs[0].tolist())})